In [ ]:
import pandas as pd

learn_data = pd.read_csv("../data/minimal_train_fs.csv", header = None)
learn_data.columns = ['Age', 'TB', 'Alkphos', 'Sgot', 'ALB', 'AR', 'BilRatio', 'Female', 'Target']
learn_data["Female"] = learn_data["Female"].astype("category")
learn_data["Target"] = learn_data["Target"].astype("category")
learn_data.head()

,Age,TB,Alkphos,Sgot,ALB,AR,BilRatio,Female,Target
0,48,1.504077,5.641907,4.304065,2.4,0.52,0.511111,0,0
1,39,0.641854,5.192957,4.127134,4.3,1.38,0.473684,0,0
2,23,0.000000,5.356586,4.382027,3.1,1.00,0.300000,0,0
3,42,-0.356675,5.023881,4.394449,3.2,1.06,0.285714,1,0
4,54,3.117950,6.324359,3.610918,3.4,0.80,0.504425,1,0


In [2]:
from sklearn.model_selection import train_test_split

X = learn_data.drop(columns = ["Target"])
Xnum = X.drop(columns = ["Female"])
y = learn_data["Target"]

X_train, X_val, Xnum_train, Xnum_val, y_train, y_val = train_test_split(X, Xnum, y, test_size = 0.33, random_state = 42)

In [3]:
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
import numpy as np

def compute_metrics (y_real, y_pred) -> list[float]:
    F1_macro = f1_score(y_real, y_pred, average = "macro")
    recall = recall_score(y_real, y_pred, average = "macro")
    prec = precision_score(y_real, y_pred, average = "macro")
    acc = accuracy_score(y_real, y_pred)
    return [F1_macro, recall, prec, acc]

def confusion (y_real, y_pred) -> None:
    TP = sum(np.logical_and(y_real == y_pred, y_real == 1))
    TN = sum(np.logical_and(y_real == y_pred, y_real == 0))
    FP = sum(np.logical_and(y_real != y_pred, y_real == 0))
    FN = sum(np.logical_and(y_real != y_pred, y_real == 1))
    print("\t\tPredicted")
    print("\t\t+1\t0")
    print(f"Real\t+1\t{TP}\t{FN}")
    print(f"\t0\t{FP}\t{TN}")
    print(f"Accuracy: {((TP + TN) / y_real.shape[0] * 100):.2f}%".format())

metrics_df = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

# Polynomial

In [4]:
from sklearn.svm import SVC

poly_model = SVC(kernel = "poly", degree = 3, gamma = "scale", class_weight = "balanced")
poly_model.fit(Xnum_train, y_train)

confusion(np.array(y_train), pd.Series(poly_model.predict(Xnum_train)))

		Predicted
		+1	0
Real	+1	75	9
	0	134	82
Accuracy: 52.33%


In [5]:
confusion(np.array(y_val), pd.Series(poly_model.predict(Xnum_val)))

		Predicted
		+1	0
Real	+1	35	7
	0	65	42
Accuracy: 51.68%


In [9]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

polysvc = SVC(kernel = "poly", gamma = "scale", class_weight = "balanced")
polysvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", polysvc)])

n = 50
m = 100
Cs = np.logspace(start = -1, stop = 2, num = n)
coef0s = np.linspace(start = -25, stop = 25, num = m)
degrees = [2, 3]

polysvc_search = GridSearchCV(estimator = polysvc_pipeline,
                              param_grid = {'svc__C' : Cs,
                                            'svc__coef0' : coef0s,
                                            'svc__degree' : degrees},
                              scoring = 'f1_macro',
                              cv = 5)
polysvc_search.fit(X, y)
polysvc_search.best_params_

KeyboardInterrupt: 